# MFE-FCGCN: the paper's GCN baseline for AD vs HC on ds004504

A clean re-implementation of **MFE-FCGCN**, the model proposed by Liu, An,
Yang & Ge (2025), *Complex & Intelligent Systems* 11:366,
doi:[10.1007/s40747-025-01974-x](https://doi.org/10.1007/s40747-025-01974-x).
The architecture follows the paper's equations (Eqs. 6–9) and hyperparameters
(pp. 7–8). `GAT_ds004504.ipynb` is this same notebook with the graph layer
swapped for attention, so their results can be compared directly.

```
for each band f in {Delta, Theta, Alpha, Beta, Gamma}:

  x_f    (19 electrodes x F_f PSD bins)
    |  BandCNN      Conv1d(1, 1, D_f),  D_f = 3, 6, 11, 71, 61        Eq. 6
    v
  (19 x 16)
    |  GCN layer 1  ReLU(E1_f . X . W1_f)    Pearson graph, 16 -> 16   Eq. 8
    v
  (19 x 16)
    |  GCN layer 2  ReLU(E2_f . H . W2_f)    MI graph,      16 -> 5    Eq. 8
    v
  (19 x 5)  --flatten-->  95                                          Eq. 7

concat 5 bands -> 475 (Eq. 9) -> MLP 30 -> 14 -> 2 -> AD / HC
```

| | paper | this notebook |
|---|---|---|
| CNN, graph layers, band concat, MLP, dropout | Eqs. 6–9, pp. 7–8 | identical (16,686 parameters) |
| loss, optimiser, batch size | cross-entropy, Adam, 64 | same |
| learning rate and schedule | 1.2e-4, halved after 10 flat epochs, stop after 20 | `LR` in Cell 2; cosine over 80 epochs, stop after 20 |
| evaluation | 80/20 split, 15-fold CV inside, 20 repeats | 5-fold grouped by subject, every subject tested once |

The last two rows are the protocol shared by both notebooks, so GCN and GAT
differ only in the graph layer.

**Paper vs released code.** The authors' repository
([brief-city/MFE-FCGCN-model](https://github.com/brief-city/MFE-FCGCN-model))
does not match the equations in four places. It uses one GCN shared by all
bands, puts MI before Pearson, uses PyG `GCNConv` (which adds self-loops and
re-normalises edges to `D^-½(A+I)D^-½`), and has no ReLU after layer 2. **This
notebook follows the paper's equations.**

**Protocol.** 5-fold cross-validation grouped by subject. Each of the 65
subjects is tested exactly once, by a model that saw none of that subject's
windows during training. About 11 held-out validation subjects decide which
epoch to keep. A subject's score is the mean probability over their windows.
The headline number is **pooled over all 65 subjects**.

**Bars to beat**, both scored on the same test subjects as this network:
- band-power logistic regression: Cell 5, refitted on this notebook's own folds;
- MFE-FCGAT (`GAT_ds004504.ipynb`): added to the Cell 9 table once that notebook has been
  run with the same protocol.

The notebook needs `subject_cache/` next to it (built by `build_cache.ipynb`;
this notebook reads its 65 AD and HC subjects). Run the cells top to bottom. **Cell 2 is normally the only
cell you need to edit.**

In [ ]:
# ===========================================================================
# CELL 1 — IMPORTS, PATHS, DEVICE
# ===========================================================================
import copy, json, math, random, time, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

WORK_DIR   = Path.cwd()
CACHE_DIR  = WORK_DIR / "subject_cache"      # point elsewhere if the cache lives elsewhere
RESULT_DIR = WORK_DIR / "results_gcn"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


print(f"torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"cache   : {CACHE_DIR}  ({len(list(CACHE_DIR.glob('sub-*.npz')))} files)")
print(f"results : {RESULT_DIR}")

In [ ]:
# ===========================================================================
# CELL 2 — CONFIGURATION  (the only cell you normally edit)
# ===========================================================================
SEED = 42

# --- data: fixed by the cache ----------------------------------------------
BAND_NAMES = ["Delta", "Theta", "Alpha", "Beta", "Gamma"]
BAND_DIMS  = {"Delta": 18, "Theta": 21, "Alpha": 26, "Beta": 86, "Gamma": 76}  # PSD bins @ 0.2 Hz
N_NODES    = 19            # electrodes
N_SUBJECTS = 65            # sub-001..036 AD (label 1), sub-037..065 HC (label 0)
N_AD       = 36

# --- model: paper pp. 7-8 ----------------------------------------------------
CNN_OUT_DIM = 16           # D_out = 16  ->  kernels 3, 6, 11, 71, 61
GCN_HIDDEN  = 16           # first graph layer: 16 neurons
GCN_OUT     = 5            # second graph layer: 5 neurons
MLP_HIDDEN  = (30, 14)     # fully connected 30 -> 14 -> 2
MLP_DROPOUT = (0.8, 0.5)

# --- training (identical to GAT_ds004504.ipynb except LR) --------------------
# The paper trains at 1.2e-4 with a plateau schedule (up to ~300 epochs). At that
# rate, results_mfagat/B_gcn_subject pooled 0.8000 acc / 0.8343 AUC. This
# notebook anneals over 80 epochs, so check 1.2e-4 against 3e-3 before settling.
LR           = 3e-3
WEIGHT_DECAY = 1e-4        # from the released code; the paper does not state one
BATCH_SIZE   = 64          # paper
MAX_EPOCHS   = 80          # also the cosine horizon: the LR anneals to ~0 by here
MIN_EPOCHS   = 15          # no checkpoint before this; the earliest epochs are noise
PATIENCE     = 20          # stop after this many epochs without improvement (paper: 20)
GRAD_CLIP    = 5.0

# --- protocol (keep identical across GCN_ds004504 and GAT_ds004504) --------
# "subject": no window from a test subject is ever used in training (honest).
# "window" : windows are shuffled freely. This reproduces the paper's ~95 %,
#            which depends on overlapping windows from the same person landing
#            on both sides of the split. Raise MAX_EPOCHS to ~300 for it.
SPLIT_LEVEL = "subject"
N_FOLDS     = 5
N_REPEATS   = 1            # use 3+ before any number goes into a write-up

assert SPLIT_LEVEL in ("subject", "window")
assert [BAND_DIMS[b] - CNN_OUT_DIM + 1 for b in BAND_NAMES] == [3, 6, 11, 71, 61]
seed_all(SEED)
print(f"MFE-FCGCN | lr {LR} | {N_REPEATS} x {N_FOLDS}-fold, {SPLIT_LEVEL}-level | "
      f"max {MAX_EPOCHS} epochs, warm-up {MIN_EPOCHS}, patience {PATIENCE}")

In [ ]:
# ===========================================================================
# CELL 3 — LOAD THE FEATURE CACHE
# ===========================================================================
# One .npz per subject; one row per 20 s window (75 % overlap):
#   x_<Band>  (n, 19, F_f)       log PSD, normalised per window     paper Eq. 1
#   e1        (n, 5, 19, 19)     |Pearson| graph, row-normalised    Eqs. 2-3
#   e2        (n, 5, 19, 19)     mutual-information graph           Eqs. 4-5
#   bp        (n, 5, 19)         mean log band power (Cells 5 and 10)
x_parts = {b: [] for b in BAND_NAMES}
e1_parts, e2_parts, bp_parts, y_parts, sid_parts = [], [], [], [], []

for i in range(1, N_SUBJECTS + 1):
    sid = f"sub-{i:03d}"
    with np.load(CACHE_DIR / f"{sid}.npz") as z:
        for b in BAND_NAMES:
            assert z[f"x_{b}"].shape[1:] == (N_NODES, BAND_DIMS[b]), (sid, b)
            x_parts[b].append(z[f"x_{b}"])
        e1_parts.append(z["e1"]); e2_parts.append(z["e2"])
        bp_parts.append(z["bp"]); y_parts.append(z["y"])
        sid_parts.append(np.full(len(z["y"]), sid))
    # a re-cached subject must never silently flip class
    assert int(y_parts[-1][0]) == (1 if i <= N_AD else 0), f"{sid}: wrong label"

X   = [np.concatenate(x_parts[b]).astype(np.float32) for b in BAND_NAMES]
E1  = np.concatenate(e1_parts).astype(np.float32)
E2  = np.concatenate(e2_parts).astype(np.float32)
BP  = np.concatenate(bp_parts).astype(np.float32)
Y   = np.concatenate(y_parts).astype(np.int64)
SID = np.concatenate(sid_parts)
del x_parts, e1_parts, e2_parts, bp_parts, y_parts, sid_parts

# The whole dataset (~180 MB) lives on the device; batches are index slices.
Xt  = [torch.from_numpy(a).to(DEVICE) for a in X]
E1t = torch.from_numpy(E1).to(DEVICE)
E2t = torch.from_numpy(E2).to(DEVICE)
Yt  = torch.from_numpy(Y).to(DEVICE)

print(f"windows  : {len(Y):,}  (AD {int((Y == 1).sum()):,} / HC {int((Y == 0).sum()):,})")
print(f"subjects : {len(np.unique(SID))}  (AD {N_AD} / HC {N_SUBJECTS - N_AD})")
for b, a in zip(BAND_NAMES, X):
    print(f"  x_{b:<6s} {a.shape}")
print(f"  e1       {E1.shape}\n  e2       {E2.shape}")

In [ ]:
# ===========================================================================
# CELL 4 — SPLITS AND METRICS
# ===========================================================================
def make_splits(level=SPLIT_LEVEL, n_folds=N_FOLDS, n_repeats=N_REPEATS, seed=SEED):
    """The outer K-fold picks the test set; 1/5 of the remainder becomes validation.

    Within each repeat, every window (and so every subject) is tested exactly once.
    """
    splits, idx = [], np.arange(len(Y))
    for r in range(1, n_repeats + 1):
        rs = seed + 1000 * r
        outer = (StratifiedGroupKFold(n_folds, shuffle=True, random_state=rs).split(idx, Y, groups=SID)
                 if level == "subject" else
                 StratifiedKFold(n_folds, shuffle=True, random_state=rs).split(idx, Y))
        for f, (trainval, test) in enumerate(outer, start=1):
            inner = StratifiedKFold(5, shuffle=True, random_state=rs + f)
            if level == "subject":
                # Hold out validation SUBJECTS, so early stopping is never
                # judged on people the model is training on.
                subs = np.unique(SID[trainval])
                labs = np.array([Y[SID == s][0] for s in subs])
                _, va = next(inner.split(subs, labs))
                is_val = np.isin(SID[trainval], subs[va])
                train, val = trainval[~is_val], trainval[is_val]
                tr_s, va_s, te_s = set(SID[train]), set(SID[val]), set(SID[test])
                assert not (tr_s & te_s or va_s & te_s or tr_s & va_s), "SUBJECT LEAKAGE"
            else:
                tr, va = next(inner.split(trainval, Y[trainval]))
                train, val = trainval[tr], trainval[va]
            splits.append(dict(repeat=r, fold=f, train=train, val=val, test=test))
    return splits


def _metrics(y, p):
    pred = (p >= 0.5).astype(int)
    return dict(acc=accuracy_score(y, pred),
                sens=recall_score(y, pred, zero_division=0),
                spec=recall_score(1 - y, 1 - pred, zero_division=0),
                f1=f1_score(y, pred, zero_division=0),
                auc=roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan)


def score(idx, prob):
    """Window-level (w_*) and subject-level (s_*) metrics.
    A subject's probability is the mean over its windows."""
    sid, y = SID[idx], Y[idx]
    subs = np.unique(sid)
    s_p = np.array([prob[sid == s].mean() for s in subs])
    s_y = np.array([y[sid == s][0] for s in subs])
    return {**{f"w_{k}": v for k, v in _metrics(y, prob).items()},
            **{f"s_{k}": v for k, v in _metrics(s_y, s_p).items()}}


SPLITS = make_splits()
_ns = lambda i: len(np.unique(SID[i]))
for s in SPLITS:
    print(f"r{s['repeat']} f{s['fold']}: train {len(s['train']):5d}w/{_ns(s['train']):2d}s | "
          f"val {len(s['val']):4d}w/{_ns(s['val']):2d}s | test {len(s['test']):4d}w/{_ns(s['test']):2d}s")
_tested = np.concatenate([s["test"] for s in SPLITS if s["repeat"] == 1])
assert len(_tested) == len(np.unique(_tested)) == len(Y)
print(f"each repeat tests all {len(Y):,} windows / {_ns(_tested)} subjects exactly once")

In [ ]:
# ===========================================================================
# CELL 5 — THE BAR: band power -> logistic regression, on the network's folds
# ===========================================================================
# Uses 95 numbers per window (5 bands x 19 electrodes), with no graph and no
# learning of structure. It is fitted on the train + validation subjects of
# every fold in SPLITS and tested on the same test subjects as the network,
# scored the same way: a subject's score is the mean of its window probabilities.
# A graph model has to beat this to justify its parameters. The network trains
# on the train subjects only (validation subjects pick its epoch), so the floor
# learns from ~25 % more subjects: the comparison favours the floor, not the network.
feat = BP.reshape(len(BP), -1)
FLOOR_OOF = np.full((N_REPEATS, len(Y)), np.nan)    # out-of-fold P(AD), one row per repeat
for sp in SPLITS:
    fit = np.concatenate([sp["train"], sp["val"]])
    clf = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, max_iter=5000))
    FLOOR_OOF[sp["repeat"] - 1, sp["test"]] = clf.fit(feat[fit], Y[fit]).predict_proba(feat[sp["test"]])[:, 1]

FLOOR = pd.DataFrame([score(np.arange(len(Y)), FLOOR_OOF[r]) for r in range(N_REPEATS)]).mean()
print(f"linear floor, subject level : acc {FLOOR['s_acc']:.4f}  recall {FLOOR['s_sens']:.4f}  "
      f"spec {FLOOR['s_spec']:.4f}  AUC {FLOOR['s_auc']:.4f}   (same folds as the network)")

---
## Model

- **Dense `E @ (W x)`, not `GCNConv`.** Every graph links all 19 electrodes,
  and E1 and E2 are already normalised by Eqs. 3 and 5. Multiplying by them
  directly applies them exactly as written. `GCNConv` would add self-loops and
  re-normalise, which is what the released code does.
- **Pearson first, MI second**, as in Eq. 8. Each band has its own weights
  (`W1_f`, `W2_f`).
- **The only difference from `GAT_ds004504.ipynb`** is `BandGCN` here versus
  `BandGAT` there. Every other cell is the same code.

In [ ]:
# ===========================================================================
# CELL 6 — MODEL  (paper Eqs. 6-9)
# ===========================================================================
class BandCNN(nn.Module):
    """Paper Eq. 6: one Conv1d(1, 1, D_f) per band, shared across the 19 electrodes.

    D_f = F_f - 16 + 1, so every band comes out 16 wide and keeps its frequency order.
    """
    def __init__(self):
        super().__init__()
        self.convs = nn.ModuleList([nn.Conv1d(1, 1, BAND_DIMS[b] - CNN_OUT_DIM + 1)
                                    for b in BAND_NAMES])

    def forward(self, x, band):                    # (B, 19, F_f) -> (B, 19, 16)
        b, n, f = x.shape
        return self.convs[band](x.reshape(b * n, 1, f)).reshape(b, n, CNN_OUT_DIM)


class BandGCN(nn.Module):
    """Paper Eq. 8:  H_f = ReLU( E2_f . ReLU( E1_f . X_f . W1_f ) . W2_f )"""
    def __init__(self):
        super().__init__()
        self.w1 = nn.Linear(CNN_OUT_DIM, GCN_HIDDEN)
        self.w2 = nn.Linear(GCN_HIDDEN, GCN_OUT)

    def forward(self, x, e1, e2):                  # x (B, 19, 16); e1, e2 (B, 19, 19)
        h = torch.relu(e1 @ self.w1(x))            # Pearson graph -> (B, 19, 16)
        return torch.relu(e2 @ self.w2(h))         # MI graph      -> (B, 19, 5)


class MFEFCGCN(nn.Module):
    """BandCNN -> per-band BandGCN -> flatten + concat 5 x 19 x 5 = 475 (Eqs. 7, 9) -> MLP."""
    def __init__(self):
        super().__init__()
        self.cnn  = BandCNN()
        self.gcns = nn.ModuleList([BandGCN() for _ in BAND_NAMES])   # W1_f, W2_f per band
        self.head = nn.Sequential(
            nn.Linear(len(BAND_NAMES) * N_NODES * GCN_OUT, MLP_HIDDEN[0]), nn.ReLU(),
            nn.Dropout(MLP_DROPOUT[0]),
            nn.Linear(MLP_HIDDEN[0], MLP_HIDDEN[1]), nn.ReLU(),
            nn.Dropout(MLP_DROPOUT[1]),
            nn.Linear(MLP_HIDDEN[1], 2))

    def forward(self, xs, e1, e2):                 # xs: 5 x (B, 19, F_f); e1, e2: (B, 5, 19, 19)
        z = [self.gcns[i](self.cnn(xs[i], i), e1[:, i], e2[:, i]).flatten(1)
             for i in range(len(BAND_NAMES))]
        return self.head(torch.cat(z, dim=1))      # logits (B, 2)


_m  = MFEFCGCN()
_xs = [torch.randn(8, N_NODES, BAND_DIMS[b]) for b in BAND_NAMES]
_e  = torch.rand(8, len(BAND_NAMES), N_NODES, N_NODES)
print(f"parameters : {sum(p.numel() for p in _m.parameters()):,}   (GAT version: 22,566)")
print(f"smoke test : logits {tuple(_m(_xs, _e, _e).shape)}")
del _m, _xs, _e

In [ ]:
# ===========================================================================
# CELL 7 — TRAINING
# ===========================================================================
def batches(idx, size=BATCH_SIZE, shuffle=False, rng=None):
    order = rng.permutation(len(idx)) if shuffle else np.arange(len(idx))
    for s in range(0, len(idx), size):
        sel = torch.as_tensor(idx[order[s:s + size]], device=DEVICE)
        yield [x[sel] for x in Xt], E1t[sel], E2t[sel], Yt[sel]


@torch.no_grad()
def predict(model, idx):
    """P(AD) for each window, in the order of idx."""
    model.eval()
    return torch.cat([torch.softmax(model(xs, e1, e2), 1)[:, 1]
                      for xs, e1, e2, _ in batches(idx, size=512)]).cpu().numpy()


def train_fold(split):
    """Train on split['train'], keep the epoch with the best validation AUC, predict test."""
    seed = SEED + 1000 * split["repeat"] + split["fold"]
    seed_all(seed)
    rng = np.random.default_rng(seed)

    model = MFEFCGCN().to(DEVICE)
    opt   = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=LR * 0.01)
    key   = "s_auc" if SPLIT_LEVEL == "subject" else "w_auc"   # select on what gets reported

    best, best_epoch, best_state, stale, history = -np.inf, 0, None, 0, []
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        lr, total = opt.param_groups[0]["lr"], 0.0
        for xs, e1, e2, y in batches(split["train"], shuffle=True, rng=rng):
            loss = F.cross_entropy(model(xs, e1, e2), y)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            total += loss.item() * len(y)
        sched.step()

        val = score(split["val"], predict(model, split["val"]))
        history.append(dict(epoch=epoch, lr=lr, train_loss=total / len(split["train"]),
                            val_w_auc=val["w_auc"], val_s_acc=val["s_acc"], val_s_auc=val["s_auc"]))

        if epoch < MIN_EPOCHS:
            continue
        if val[key] > best:
            best, best_epoch, stale = val[key], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            stale += 1
            if stale >= PATIENCE:
                break

    model.load_state_dict(best_state)
    return dict(best_epoch=best_epoch, epochs_run=epoch,
                test_prob=predict(model, split["test"]), history=pd.DataFrame(history))


print("training functions ready")

In [ ]:
# ===========================================================================
# CELL 8 — CROSS-VALIDATION
# ===========================================================================
rows, HISTORY = [], {}
OOF = np.full((N_REPEATS, len(Y)), np.nan)        # out-of-fold P(AD), one row per repeat
t0 = time.time()

for sp in SPLITS:
    ts = time.time()
    r = train_fold(sp)
    OOF[sp["repeat"] - 1, sp["test"]] = r["test_prob"]
    m = score(sp["test"], r["test_prob"])
    rows.append(dict(repeat=sp["repeat"], fold=sp["fold"], best_epoch=r["best_epoch"],
                     epochs_run=r["epochs_run"], seconds=round(time.time() - ts), **m))
    HISTORY[(sp["repeat"], sp["fold"])] = r["history"]
    r["history"].to_csv(RESULT_DIR / f"history_r{sp['repeat']}_f{sp['fold']}.csv", index=False)
    print(f"r{sp['repeat']} f{sp['fold']}  best epoch {r['best_epoch']:2d}/{r['epochs_run']:2d}  "
          f"window acc {m['w_acc']:.4f} auc {m['w_auc']:.4f}  |  "
          f"subject acc {m['s_acc']:.4f} auc {m['s_auc']:.4f}  [{rows[-1]['seconds']}s]", flush=True)

FOLDS = pd.DataFrame(rows)
FOLDS.to_csv(RESULT_DIR / "fold_results.csv", index=False)
np.save(RESULT_DIR / "oof_prob.npy", OOF)

# Pooled out-of-fold score: each repeat scores all 65 subjects once, each by a
# model that never saw them. This is the number to quote.
POOLED = pd.DataFrame([score(np.arange(len(Y)), OOF[r]) for r in range(N_REPEATS)])
P    = POOLED.mean()
P_SD = POOLED.std(ddof=1) if N_REPEATS > 1 else P * 0
pm = lambda k: f"{P[k]:.4f}" + (f" +- {P_SD[k]:.4f}" if N_REPEATS > 1 else "")

print(f"\nPOOLED  window  acc {pm('w_acc')}  auc {pm('w_auc')}")
print(f"POOLED  subject acc {pm('s_acc')}  auc {pm('s_auc')}  "
      f"sens {P['s_sens']:.4f}  spec {P['s_spec']:.4f}   [{(time.time() - t0) / 60:.1f} min]")

(RESULT_DIR / "summary.json").write_text(json.dumps(dict(
    model="MFE-FCGCN", pooled=P.to_dict(), pooled_sd=P_SD.to_dict(), floor=FLOOR.to_dict(),
    config=dict(LR=LR, WEIGHT_DECAY=WEIGHT_DECAY, BATCH_SIZE=BATCH_SIZE, MAX_EPOCHS=MAX_EPOCHS,
                MIN_EPOCHS=MIN_EPOCHS, PATIENCE=PATIENCE, GRAD_CLIP=GRAD_CLIP,
                CNN_OUT_DIM=CNN_OUT_DIM, GCN_HIDDEN=GCN_HIDDEN, GCN_OUT=GCN_OUT,
                MLP_HIDDEN=MLP_HIDDEN, MLP_DROPOUT=MLP_DROPOUT,
                SPLIT_LEVEL=SPLIT_LEVEL, N_FOLDS=N_FOLDS, N_REPEATS=N_REPEATS, SEED=SEED)), indent=2))

In [ ]:
# ===========================================================================
# CELL 9 — RESULTS
# ===========================================================================
# The other architecture is shown too if its notebook has been run next to this
# one with the same protocol (same splits, so the comparison is like-for-like).
OTHER_DIR = WORK_DIR / "results_gat"
other = (json.loads((OTHER_DIR / "summary.json").read_text())
         if (OTHER_DIR / "summary.json").exists() else None)
if other and any(other["config"][k] != v for k, v in
                 dict(SEED=SEED, N_FOLDS=N_FOLDS, N_REPEATS=N_REPEATS, SPLIT_LEVEL=SPLIT_LEVEL).items()):
    print(f"note: {OTHER_DIR.name}/ used a different protocol, so it is left out\n")
    other = None

table = [(f"MFE-FCGCN (this run, lr {LR:g})", P)]
if other:
    table.append((f"MFE-FCGAT ({OTHER_DIR.name}, lr {other['config']['LR']:g})", pd.Series(other["pooled"])))

print("=" * 74)
print(f"{'subject level, pooled over 65':38s}{'acc':>9s}{'AUC':>9s}{'sens':>9s}{'spec':>9s}")
print("-" * 74)
for name, p in table:
    print(f"{name:38s}{p['s_acc']:9.4f}{p['s_auc']:9.4f}{p['s_sens']:9.4f}{p['s_spec']:9.4f}")
print(f"{'band-power LR, same folds':38s}{FLOOR['s_acc']:9.4f}{FLOOR['s_auc']:9.4f}"
      f"{FLOOR['s_sens']:9.4f}{FLOOR['s_spec']:9.4f}")
print("=" * 74)
if not other:
    print("run GAT_ds004504.ipynb to add MFE-FCGAT to this table")
print(f"vs linear floor: acc {P['s_acc'] - FLOOR['s_acc']:+.4f}   AUC {P['s_auc'] - FLOOR['s_auc']:+.4f}"
      "   (under ~0.03 is noise at N_REPEATS = 1)")

# --- was the learning rate sensible? -----------------------------------------
print(f"\nbest epochs: {FOLDS.best_epoch.tolist()}   (warm-up {MIN_EPOCHS}, cap {MAX_EPOCHS})")
if (FOLDS.best_epoch == MIN_EPOCHS).mean() >= 0.6:
    print("  -> most folds peaked at the warm-up floor: the model overshoots early; try a lower LR")
elif (FOLDS.epochs_run == MAX_EPOCHS).mean() >= 0.6:
    print("  -> most folds were still improving at the cap: try a higher LR or more epochs")

# --- which subjects are wrong? ------------------------------------------------
_p = np.nanmean(OOF, axis=0)
PER_SUBJECT = pd.DataFrame([dict(subject=s, label="AD" if Y[SID == s][0] else "HC",
                                 p_ad=_p[SID == s].mean()) for s in np.unique(SID)])
PER_SUBJECT["called"]  = np.where(PER_SUBJECT.p_ad > 0.5, "AD", "HC")
PER_SUBJECT["correct"] = PER_SUBJECT.called == PER_SUBJECT.label
PER_SUBJECT.to_csv(RESULT_DIR / "per_subject.csv", index=False)
print(f"\n{PER_SUBJECT.correct.sum()}/{len(PER_SUBJECT)} subjects correct; misclassified:")
print(PER_SUBJECT[~PER_SUBJECT.correct].sort_values("p_ad").to_string(
    index=False, float_format=lambda v: f"{v:.3f}"))

# --- plots --------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
_y = (PER_SUBJECT.label == "AD").astype(int)
ax[0].plot(*roc_curve(_y, PER_SUBJECT.p_ad)[:2], lw=2,
           label=f"MFE-FCGCN  AUC {roc_auc_score(_y, PER_SUBJECT.p_ad):.3f}")
if other:
    _o  = np.nanmean(np.load(OTHER_DIR / "oof_prob.npy"), axis=0)
    _op = np.array([_o[SID == s].mean() for s in PER_SUBJECT.subject])
    ax[0].plot(*roc_curve(_y, _op)[:2], lw=1.5, label=f"MFE-FCGAT  AUC {roc_auc_score(_y, _op):.3f}")
_f  = np.nanmean(FLOOR_OOF, axis=0)
_fp = np.array([_f[SID == s].mean() for s in PER_SUBJECT.subject])
ax[0].plot(*roc_curve(_y, _fp)[:2], lw=1.2, ls="--", label=f"linear floor  AUC {roc_auc_score(_y, _fp):.3f}")
ax[0].plot([0, 1], [0, 1], "k:", lw=0.8)
ax[0].set(xlabel="1 - specificity", ylabel="sensitivity", title="Subject-level ROC (65 subjects)")
ax[0].legend(); ax[0].grid(alpha=0.3)

for (r, f), h in HISTORY.items():
    ax[1].plot(h.epoch, h.val_s_auc, lw=1.2, label=f"r{r} f{f}")
ax[1].axvline(MIN_EPOCHS, color="grey", ls=":", lw=1)
ax[1].set(xlabel="epoch", ylabel="validation subject AUC", title="MFE-FCGCN: validation AUC per fold")
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESULT_DIR / "summary.png", dpi=140)
plt.show()

---
## Ensemble: network + band power (soft vote)

The graph network and the band-power logistic regression make different
mistakes, so averaging their probabilities helps. The weights are equal and
fixed in advance, so nothing is tuned on test subjects. Both models are scored
on the same test subjects. The cell reads the network's saved out-of-fold
probabilities, so after a restart it only needs Cells 1–5, not another training
run.

In [ ]:
# ===========================================================================
# CELL 10 — ENSEMBLE: network + band-power floor (soft vote, no extra training)
# ===========================================================================
# FLOOR_OOF comes from Cell 5 (the network's own folds), so all three rows
# below are scored on identical test subjects.

NET_OOF = np.load(RESULT_DIR / "oof_prob.npy")
assert NET_OOF.shape == FLOOR_OOF.shape, "run Cell 8 with the same N_REPEATS first"
ENS_OOF = (NET_OOF + FLOOR_OOF) / 2
np.save(RESULT_DIR / "ensemble_oof_prob.npy", ENS_OOF)

ENS = {}
print("=" * 86)
print(f"{'pooled over 65 subjects':38s}{'acc':>8s}{'recall':>8s}{'spec':>8s}{'AUC':>8s}{'win acc':>8s}{'win AUC':>8s}")
print("-" * 86)
for name, oof in [(f"this network (lr {LR:g})", NET_OOF), ("band-power LR, same folds", FLOOR_OOF),
                  ("soft vote: network + band power", ENS_OOF)]:
    m = pd.DataFrame([score(np.arange(len(Y)), oof[r]) for r in range(N_REPEATS)]).mean()
    ENS[name] = m
    print(f"{name:38s}" + "".join(f"{m[c]:8.4f}" for c in ["s_acc", "s_sens", "s_spec", "s_auc", "w_acc", "w_auc"]))
print("=" * 86)
_ = (RESULT_DIR / "ensemble_summary.json").write_text(json.dumps({k: v.to_dict() for k, v in ENS.items()}, indent=2))

---
## Reading the result

- **Quote the pooled row only.** Each fold has 13 test subjects, so a single
  subject is worth 7.7 accuracy points. Single-fold numbers in this project
  came out up to 0.09 too optimistic.
- **Trust AUC over accuracy.** Accuracy uses a fixed 0.5 cut, and calibration
  shifts between groups of subjects. Tuning the cut on validation subjects was
  tested and made results worse.
- **At `N_REPEATS = 1`, AUC differences under ~0.03 are noise.** That includes
  the GCN-vs-GAT gap. Set `N_REPEATS = 3` in both notebooks before any number
  goes into a write-up.
- **The learning rate matters more than the architecture on this data.**
  Compare the two models each at its own best `LR`. The best-epoch check in
  Cell 9 tells you which direction to move it.
- `SPLIT_LEVEL = "window"` reproduces the paper's protocol (about 0.95). That
  score comes from overlapping windows of the same person landing in both train
  and test, so report it only with that label.